# JT script for chem space mapping 

## Imports 

In [ ]:
import pandas as pd 
import numpy as np 

import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import hdbscan
import umap.umap_ as umap
import umap.plot
import seaborn as sns
import plotly.express as px 
import plotly.graph_objects as go
import molplotly



# rdkit
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

# vendi shit
import torch 
from vendi_score import vendi
from vendi_score import molecule_utils

## Load Data

In [ ]:
feature_df = pd.read_excel('six_membered_w_sim_scores.xlsx', index_col=0)
smiles_df = feature_df
feature_df = feature_df.drop(['SMILES', 'mol_id', 'id', 'mol_id.1', 'dice_sim_r_3', 'dice_sim_r_2'], axis=1)
display(feature_df)
display(smiles_df)

## Colinearity & Scaling

In [ ]:
threshold = 0.8 #set the desired colinearity cut off
######################
df_corr = feature_df.corr()
df_not_correlated = ~(df_corr.mask(np.tril(np.ones([len(df_corr)]*2, dtype=bool))).abs() > threshold).any()
un_corr_idx = df_not_correlated.loc[df_not_correlated[df_not_correlated.index] == True].index
feature_df = feature_df[un_corr_idx]
print(f'Shape of descriptors file after removing parameters with R^2 > {threshold} : ',feature_df.shape)
# feature_df = feature_df.drop(["('acids', 'mol', 'dipole', 'min_E')", "('acids', 'mol', 'dipole', 'min')", "('acids', 'mol', 'HOMO', 'boltz')", "('acids', 'mol', 'dipole', 'boltz')"], axis =1)
display(feature_df)

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(feature_df)
display(features_scaled)
features_scaled.shape

## PCA

In [ ]:

n_components=4 # set your number of PCA components
#######################
pca = PCA(n_components)
pca_results = pca.fit_transform(features_scaled)
pca_score = pca.explained_variance_ratio_


print('Total variance explained:', round(np.sum(pca_score*100), 1), '%\n')
# print(pca_feature_score)

pca_df = pd.DataFrame(pca_results, columns =[f'PC{i+1}' for i in range(n_components)])

plt.figure(figsize=(8,6))
plt.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.7, color='royalblue', marker='o', s=50)
plt.xlabel('PC1', fontsize = 12)
plt.ylabel('PC2', fontsize = 12)
plt.xticks([])
plt.yticks([])

plt.show()

### Determine PCA Feature Contributions

In [ ]:
feature_names = feature_df.columns
# display(feature_names)
# display(len(feature_names))

feature_contribs = pd.DataFrame(pca.components_.T,
                                index=feature_names,
                                columns=[f'PC{i+1}' for i in range(n_components)])
# .T transposes the data so that instead of having th PCA component as (n_compenents, n_features; aka the features as columns and PCAs as rows), you have the opposite(features as the columns and PCs as the columns)
display(feature_contribs)

# option to save to excel file
# feature_contribs.to_excel('PCA_contributions.xlsx')

### 3D PCA plot

In [ ]:
fig = plt.figure(figsize=(10,6))

ax = fig.add_subplot(111, projection='3d')
ax.scatter(pca_df['PC1'], pca_df['PC2'], pca_df['PC3'],alpha=0.7, color='royalblue', marker='o', s=50)


ax.figure(figsize=(10,6))
ax.scatter(pca_df['PC1'], pca_df['PC2'], pca_df['PC3'], alpha=0.7, color='royalblue', marker='o', s=50)
ax.xlabel('PC1', fontsize = 12)
ax.ylabel('PC2', fontsize = 12)
ax.zlabel('PC3', fontsize=12)
ax.xticks([])
ax.yticks([])
ax.zticks([])

plt.show()

### Visualize PCA space by variable

In [ ]:


# visualizes the PCA space by a color variable; can change to do size as well by uncommenting those rows
# size = vbur["vbur_norm"]
color=feature_df["('acids', 'atom', 'C4_V_bur', 'max')"] #name of given feature
# color = vbur['ir_norm']
fig = go.Figure(data=[go.Scatter(
    x=pca_df['PC1'], y=pca_df['PC2'], 
    mode='markers',
    marker=dict(
        size=5,
        # sizemode='area',
        # sizeref=2.*max(size)/(25.**2),
        # sizemin=4,
        color=color,
        showscale=True
    )
)])

fig.update_layout(
    autosize=False,
    width=1000,
    height=800,
    xaxis_title='PC1',
    yaxis_title='PC2'
)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

# display(size)
fig.show()

## Kmeans clustering

In [ ]:

n_clusters = 30 #set your desired number of clusters 

##################

kmeans = KMeans(n_clusters, random_state=42)
clusters = kmeans.fit_predict(pca_df)
display(clusters)
pca_df['cluster']=clusters #adds cluster numbers to the pca_df
# clusters = kmeans.fit_predict(coord)
# display(clusters)


plt.figure(figsize=(8,6))
sns.scatterplot(pca_df, 
                x=pca_df['PC1'], 
                y=pca_df['PC2'], 
                alpha=0.7, hue='cluster', 
                palette = 'Paired', 
                marker ='o', 
                s =  50)
plt.xticks([])
plt.yticks([])
plt.show()

## HDBSCAN clustering

In [ ]:
coordinates = pca_df[['PC1', 'PC2']]
display(coordinates)

In [ ]:
# HDBSCAN hyperparamters:
## https://hdbscan.readthedocs.io/en/latest/parameter_selection.html

clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=75, min_samples = 10, cluster_selection_method='leaf')
clusters = clusterer.fit_predict(coordinates)
display(clusters)
pca_df['hdbscan_cluster']=clusters

clustered_points = (clusters != -1)  # points that are not noise
print("Fraction of points clustered:",np.sum(clustered_points) / feature_df.shape[0])


plt.figure(figsize=(8,6))
sns.scatterplot(pca_df, x=pca_df['PC1'], y=pca_df['PC2'], alpha=0.7, hue='hdbscan_cluster', palette = 'Paired', marker ='o', s =  50)
plt.xticks([])
plt.yticks([])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.show()
display(pca_df)